# Notebook 5: Forward Propagation

In the previous notebooks, we looked at a single neuron. Now, we will explore how multiple neurons work together in layers to transform an input into a prediction. This process is called **Forward Propagation**.

**Learning Methodology:**
**Concept $\rightarrow$ Manual Calculation $\rightarrow$ PyTorch Implementation $\rightarrow$ Interpretation**

## 1. What is Forward Propagation?

**Explanation:**
Forward propagation is the process where input data travels through the neural network from the input layer, through one or more hidden layers, and finally to the output layer. At each step, the network performs linear transformations (weights and biases) and non-linear transformations (activations).

**The Flow:**
`Input` $\rightarrow$ `Hidden Layer(s)` $\rightarrow$ `Output Layer` $\rightarrow$ `Prediction`

**Components:**
- **Input Layer:** The raw data entering the network.
- **Hidden Layer:** Layers that extract features. They perform $z = W \cdot x + b$, then apply an activation function.
- **Output Layer:** The final layer that produces the result (e.g., a class probability).
- **Weights & Biases:** The learned parameters that determine the strength and offset of the signal.

## 2. Manual Calculation using NumPy

Let's calculate the output of a tiny network manually:
- **Input:** 2 features ($x_1, x_2$)
- **Hidden Layer:** 3 neurons
- **Output Layer:** 1 neuron
- **Activation:** ReLU for hidden, Sigmoid for output.

**Step-by-Step Math:**
1.  **Hidden Layer:** $H = \text{ReLU}(X \cdot W_{hidden} + b_{hidden})$
2.  **Output Layer:** $\hat{y} = \text{Sigmoid}(H \cdot W_{output} + b_{output})$

In [ ]:
import numpy as np

def relu(x):
    return np.maximum(0, x)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# 1. Setup Data
X = np.array([1.5, -2.0]) # Input (1x2)

# 2. Hidden Layer Parameters (3 neurons)
W_hidden = np.array([
    [0.2, 0.5, -0.1], # Weights for x1
    [0.8, -0.3, 0.4]   # Weights for x2
]) # Shape (2, 3)
b_hidden = np.array([0.1, -0.2, 0.5]) # Shape (3,)

# 3. Output Layer Parameters (1 neuron)
W_output = np.array([0.7, -0.5, 0.2]) # Shape (3,)
b_output = -0.1

# --- CALCULATION ---
print("--- Step 1: Hidden Layer ---")
z_hidden = np.dot(X, W_hidden) + b_hidden
a_hidden = relu(z_hidden)
print(f"Linear Sum (z): {z_hidden}")
print(f"Activated (a): {a_hidden}")

print("\n--- Step 2: Output Layer ---")
z_output = np.dot(a_hidden, W_output) + b_output
prediction = sigmoid(z_output)
print(f"Linear Sum (z): {z_output}")
print(f"Final Prediction: {prediction}")

## 3. Implementation using PyTorch

Now we implement the exact same architecture using PyTorch's `nn.Module`. PyTorch handles the matrix multiplication and weight management automatically.

In [ ]:
import torch
import torch.nn as nn

class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        # Linear layer 1: 2 inputs -> 3 outputs
        self.hidden = nn.Linear(2, 3)
        # Linear layer 2: 3 inputs -> 1 output
        self.output = nn.Linear(3, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.hidden(x))
        x = self.sigmoid(self.output(x))
        return x

# Initialize model and input
model = SimpleNet()
X_torch = torch.tensor([1.5, -2.0], dtype=torch.float32)

# Set weights manually to match NumPy example for comparison
with torch.no_grad():
    model.hidden.weight.copy_(torch.tensor([[0.2, 0.8], [0.5, -0.3], [-0.1, 0.4]])) # Note: PyTorch stores weights transposed (out, in)
    model.hidden.bias.copy_(torch.tensor([0.1, -0.2, 0.5]))
    model.output.weight.copy_(torch.tensor([[0.7, -0.5, 0.2]]))
    model.output.bias.copy_(torch.tensor([-0.1]))

prediction = model(X_torch)
print(f"PyTorch Prediction: {prediction.item():.4f}")

## 4. How does an input finally become a prediction?

**The Summary:**
1.  **Linear Transformation:** The input is multiplied by weights and shifted by a bias. This is essentially a "weighted vote" where the neuron decides which features are important.
2.  **Non-Linear Activation:** The result is passed through a function (like ReLU). This allows the network to learn complex patterns instead of just straight lines.
3.  **Layer Stacking:** The output of the first layer becomes the input for the second. Each layer builds a more complex understanding (e.g., Pixels $\rightarrow$ Edges $\rightarrow$ Shapes $\rightarrow$ Object).
4.  **Final Projection:** The last layer compresses all the learned features into a single value (or a vector for multiclass) and squashes it into a range (like 0 to 1) using Sigmoid or Softmax to give us a probability.